In [1]:
!pip install mlflow onnx onnxruntime fastapi uvicorn scikit-learn pandas joblib -q
print("All production libraries installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.

In [2]:
%%writefile data_loader.py
import pandas as pd

def load_data(path: str):
    return pd.read_csv(path)

Writing data_loader.py


In [4]:
%%writefile feature_pipeline.py
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

def get_preprocessing_pipeline(num_cols, cat_cols):
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_cols),
            ('cat', cat_transformer, cat_cols)
        ]
    )
    return preprocessor

Overwriting feature_pipeline.py


In [6]:
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Generate mock data for demonstration
X_m, y_m = make_classification(n_samples=500, n_features=5, random_state=42)
df_train = pd.DataFrame(X_m, columns=[f"feat_{i}" for i in range(5)])
df_train['target'] = y_m
df_train.to_csv("mock_data.csv", index=False)

def train_and_save_model():
    df = pd.read_csv("mock_data.csv")
    X = df.drop(columns=['target'])
    y = df['target']

    model = RandomForestClassifier(n_estimators=50, random_state=42)
    model.fit(X, y)
    joblib.dump(model, "model.joblib")
    print("Model trained and saved as model.joblib")

train_and_save_model()

# Required end-to-end validation check function
def end_to_end_check(model_path, data_path, n_batch=20):
    model = joblib.load(model_path)
    df = pd.read_csv(data_path)

    # Sample a batch of rows
    batch_df = df.sample(n=n_batch, random_state=42)
    if 'target' in batch_df.columns:
        X_batch = batch_df.drop(columns=['target'])
    else:
        X_batch = batch_df

    preds = model.predict(X_batch)
    probs = model.predict_proba(X_batch)

    # Assertions for validation
    assert set(preds).issubset({0, 1}), "Predictions must be strictly binary {0, 1}"
    assert np.all((probs >= 0.0) & (probs <= 1.0)), "Probabilities must be between 0.0 and 1.0"

    print("End-to-End Pipeline Check PASSED successfully!")
    return preds, probs

# Execute the check
end_to_end_check("model.joblib", "mock_data.csv", n_batch=20)

Model trained and saved as model.joblib
End-to-End Pipeline Check PASSED successfully!


(array([0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0]),
 array([[1.  , 0.  ],
        [0.1 , 0.9 ],
        [0.78, 0.22],
        [0.  , 1.  ],
        [1.  , 0.  ],
        [0.  , 1.  ],
        [0.  , 1.  ],
        [1.  , 0.  ],
        [1.  , 0.  ],
        [0.  , 1.  ],
        [0.  , 1.  ],
        [0.3 , 0.7 ],
        [1.  , 0.  ],
        [0.  , 1.  ],
        [0.96, 0.04],
        [0.  , 1.  ],
        [0.98, 0.02],
        [1.  , 0.  ],
        [1.  , 0.  ],
        [1.  , 0.  ]]))

In [9]:
%%writefile main.py
from fastapi import FastAPI
import joblib
import pandas as pd

app = FastAPI()
model = joblib.load("model.joblib")

@app.post("/predict")
def predict_endpoint(data: dict):
    df = pd.DataFrame([data])
    pred = model.predict(df)
    prob = model.predict_proba(df).max()
    return {"prediction": int(pred[0]), "probability": float(prob)}

print("FastAPI main.py created successfully!")

Overwriting main.py
